# Q6 · Dự đoán KSI hỗ trợ điều phối cấp cứu

**Phụ trách** A. Đánh giá khả năng dự đoán vụ Fatal/Serious khi nhận cuộc gọi 999. [Lộ trình Q6](../question.md)

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

# Cho phép import src/ khi kernel chạy trong thư mục notebooks/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from src.data_prep.road_safety import preview_table

inputs = {
    "accidents_clean.csv": ['accident_id', 'accident_date', 'accident_year', 'is_ksi', 'accident_hour', 'accident_weekday', 'accident_month', 'latitude', 'longitude', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'urban_or_rural_area', 'light_conditions', 'weather_conditions', 'road_surface_conditions'],
}

# Chỉ đọc mẫu để kiểm tra schema. Dùng toàn bộ mẫu hợp lệ khi phân tích.
for filename, required_columns in inputs.items():
    data_path = processed_dir / filename
    if not data_path.is_file():
        print(f"Chưa có {filename}. Chạy notebook 01 để tạo bảng.")
        continue
    preview_table(data_path, required_columns)
    print(f"Đã kiểm tra cột đầu vào của {filename}.")

## 1. Nhãn và thông tin sẵn có khi gọi

`is_ksi` là nhãn từ `Accident_Severity` (1/2 là KSI, 3 là Slight). Giữ khóa và ngày để kiểm tra, chia tập. Loại nhãn gốc, mã vụ, số người bị nạn, chi tiết thương tích và thông tin cảnh sát hậu kiểm khỏi X. Số phương tiện và cờ loại người tham gia chỉ vào thí nghiệm bổ sung khi có căn cứ chúng được ghi nhận tại thời điểm gọi. Cờ hồi cứu từ hồ sơ người bị nạn không mặc định là đặc trưng cuộc gọi.

In [ ]:
# Xác nhận các cột thực sự biết được lúc nhận tin và ghi lại danh sách đưa vào X.

## 2. Chia tập theo thời gian

Huấn luyện 2005–2013, kiểm tra 2014–2015. Sắp xếp theo ngày, kiểm tra không giao nhau về vụ và dùng CV tiến thời gian trong train. Báo cỡ mẫu và tỷ lệ KSI thực tế.

In [ ]:
# Chuẩn bị các tập thời gian và kiểm tra thứ tự ngày, khóa vụ và nhãn.

## 3. Baseline và hai mô hình

So sánh DummyClassifier, LogisticRegression và một mô hình phi tuyến. Đóng gói điền thiếu, mã hóa và scale khi cần trong ColumnTransformer và Pipeline.

In [ ]:
# Tạo pipeline và chọn tham số/ngưỡng chỉ với tập huấn luyện theo thời gian.

## 4. Đánh giá test và sai sót

Sau khi khóa mô hình và ngưỡng, báo average precision, recall và F1 trên test. Trình bày precision–recall, phân tích FN/FP theo tốc độ, khu vực và nhóm có dữ liệu. Chọn vài trường hợp để kiểm tra sai sót.

In [ ]:
# Lưu bảng điểm, đồ thị và thống kê sai số sau khi đánh giá.

## 5. Diễn giải và giới hạn sử dụng

Dùng permutation importance với tập đã định trước và đối chiếu hệ số mô hình tuyến tính. Viết nhận xét từ kết quả ngoài mẫu, ghi các điều kiện cần xác minh trước khi áp dụng cho điều phối.